# Lab 2: Reading Python You Didn't Write
**BUS-G 492 · Coding with AI for Business Analysts**

In [ ]:
# --- Setup: run this cell first -------------------------------------------
# Finds the course data whether you are in Colab or on your own laptop.
# INSTRUCTOR: set DATA_URL once to the raw GitHub folder that holds /data (see instructor README).
import os
DATA_URL = "https://raw.githubusercontent.com/YOUR-ACCOUNT/g492-coding-with-ai/main/data/"
for candidate in ["data/", "../data/", "../../data/"]:
    if os.path.exists(candidate + "orders.csv"):
        DATA = candidate
        break
else:
    DATA = DATA_URL
print("Reading data from:", DATA)

**Goal today:** get fast at *reading* pandas code, because AI will write most of it for you.

Every exercise follows the same loop:
1. **Predict**: read the code and write down what you expect *before* running it (a number, a shape, "error").
2. **Run** it.
3. **Explain**: if you were wrong, find out why. Asking the AI "explain this line by line" is allowed *after* you predict.

Five snippets below were "written by an AI". **Four of them contain a bug, and only one of those bugs crashes.**

In [ ]:
import pandas as pd
orders = pd.read_csv(DATA + "orders.csv")            # note: no parse_dates
items = pd.read_csv(DATA + "order_items.csv")
products = pd.read_csv(DATA + "products.csv")
customers = pd.read_csv(DATA + "customers.csv")
returns = pd.read_csv(DATA + "returns.csv")
print(len(orders), len(items), len(customers), len(returns))

## Warm-up: the 8 verbs (Excel → pandas)

| You want to… | Excel | pandas |
|---|---|---|
| keep some rows | Filter | `df[df.col == x]` |
| keep some columns | hide columns | `df[["a", "b"]]` |
| add a calculated column | new formula column | `df["c"] = df.a * df.b` |
| summarize by group | PivotTable | `df.groupby("g").col.sum()` |
| look up from another table | XLOOKUP | `df.merge(other, on="key")` |
| sort | Sort | `df.sort_values("col")` |
| count rows | COUNTA | `len(df)`, `df.shape` |
| save | Save As | `df.to_csv(...)`, `df.to_excel(...)` |

## Snippet A: a working example (predict the shape)
```python
a = (orders[orders.status == "Completed"]
       .groupby(["country", "channel"])
       .order_id.count()
       .unstack())
```
**Predict:** how many rows and columns will `a` have? What does each cell mean?

In [ ]:
prediction_A = ""   # e.g. "6 rows (countries) x 3 columns (channels); cells = # completed orders"
a = (orders[orders.status == "Completed"].groupby(["country", "channel"]).order_id.count().unstack())
a

## Snippet B: "US orders from the website"
```python
web_us = orders[orders.country == "United States" & orders.channel == "Web"]
```
**Predict:** how many rows? (Or will it fail?)

In [ ]:
prediction_B = ""
web_us = orders[orders.country == "United States" & orders.channel == "Web"]

This was the **loud** bug: it crashed, so you couldn't miss it. The rest are **silent**. They run, and they give you a number that looks fine.

## Snippet C: "Q2 2026 orders"
```python
q2 = orders[(orders.order_date >= "2026-4-1") & (orders.order_date <= "2026-6-30")]
```
**Predict:** roughly how many orders were placed in Q2 2026? (Hint: about 20,700 orders over 18 months, and Q2 is a busy season.)

In [ ]:
prediction_C = ""
q2 = orders[(orders.order_date >= "2026-4-1") & (orders.order_date <= "2026-6-30")]
len(q2)

**Explain in one sentence** why the result was wrong. (Try `orders.dtypes` and `"2026-05-10" >= "2026-4-1"`.)

## Snippet D: "Revenue and returns by product"
```python
lines = items.merge(returns, on=["order_id", "product_id"], how="left")
lines["revenue"] = lines.quantity * lines.unit_price
print(len(items), len(lines))
```
**Predict:** will `len(lines)` equal `len(items)`? (A left join keeps every row on the left… right?)

In [ ]:
prediction_D = ""
lines = items.merge(returns, on=["order_id", "product_id"], how="left")
lines["revenue"] = lines.quantity * lines.unit_price
print(len(items), len(lines))

**Habit to steal:** `merge(..., validate="one_to_one")` or `"many_to_one"` makes pandas *crash* instead of silently duplicating rows. Turn silent bugs into loud ones.

## Snippet E: "Revenue by customer segment"
```python
seg = (orders.merge(items, on="order_id").merge(customers, on="customer_id")
         .assign(rev=lambda d: d.quantity * d.unit_price)
         .groupby("segment").rev.sum())
print(seg, seg.sum())
```
**Predict:** will `seg.sum()` equal the total of `quantity * unit_price` over all items?

In [ ]:
prediction_E = ""
seg = (orders.merge(items, on="order_id").merge(customers, on="customer_id")
         .assign(rev=lambda d: d.quantity * d.unit_price).groupby("segment").rev.sum())
print(seg, "\nsum of segments:", round(seg.sum()), " total:", round((items.quantity * items.unit_price).sum()))

## Snippet F: "What's our average discount?"
```python
avg_disc = items.groupby("order_id").discount_pct.mean().mean()
```
**Predict:** is this the average discount *per dollar sold*? Per order? Per line? Which one would Finance want?

In [ ]:
prediction_F = ""
avg_disc = items.groupby("order_id").discount_pct.mean().mean()
avg_disc

## Part 2: Your turn to be the reviewer (15 min)
Ask your AI assistant:
> *Using orders.csv, order_items.csv and customers.csv, compute each acquisition channel's average net revenue per customer for customers who signed up in 2025.*

Then **before running**, annotate every line of its code with a comment in your own words, and mark each line where it made an assumption (e.g., which customers count, what "net" means, currency, customers with zero orders).
Run it, then write the two assumptions most likely to change the answer.

In [ ]:
# Paste AI code here, with YOUR comment on every line

### Exit ticket (submit)
1. Which snippet's bug would have been **most dangerous** in a real report? Why?
2. Write one habit you'll use every time AI gives you pandas code.